# 📚 Regression Analysis: A Complete Beginner's Guide

## Welcome to This Lesson!

This notebook will teach you everything you need to know about regression analysis. By the end, you'll understand:

1. **What regression is** and why we use it
2. **How to interpret** all the numbers in regression output
3. **Different types** of regression and when to use each
4. **How to evaluate** if your model is good

---

## 🎯 Part 1: What is Regression?

### The Big Picture

**Regression** is a way to understand and predict relationships between variables.

**Real-world example:** You want to predict house prices. What affects the price?
- Number of bedrooms
- Square footage
- Location
- Age of the house

Regression helps us answer: *"How much does each factor contribute to the price?"*

### Key Terms

| Term | Also Called | What It Means | Example |
|------|-------------|---------------|----------|
| **Target Variable** | Dependent Variable, Y, Response | What we want to predict | House Price |
| **Predictor Variables** | Independent Variables, X, Features | What we use to predict | Bedrooms, Size |
| **Coefficients** | Weights, Betas (β) | How much each predictor affects the target | Each bedroom adds $10,000 |

### The Basic Formula

```
Y = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ + ε
```

Where:
- **Y** = What we're predicting (house price)
- **β₀** = Intercept (base price when all X's are 0)
- **β₁, β₂, etc.** = Coefficients (how much each X affects Y)
- **X₁, X₂, etc.** = Our predictor variables
- **ε** = Error (what we can't explain)

---

## 🛠️ Part 2: Setting Up Our Environment

First, let's import the tools we need. Each library has a specific purpose:

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

# pandas: Think of it as Excel for Python
# - Handles data in tables (called DataFrames)
import pandas as pd

# numpy: For mathematical operations
# - Fast calculations on arrays of numbers
import numpy as np

# statsmodels: Statistical modeling library
# - Gives us DETAILED regression statistics
# - Shows p-values, confidence intervals, etc.
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# sklearn: Machine learning library
# - Good for predictions and model validation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look nicer
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

## 📊 Part 3: Loading and Understanding Our Data

We'll use the **Boston Housing Dataset** - a classic dataset for learning regression.

**Our Goal:** Predict the median value of homes (MEDV) based on various neighborhood characteristics.

In [ ]:
# ============================================
# LOAD THE DATA
# ============================================

# Load the Boston Housing dataset
# ⚠️ UPDATE THIS PATH to match your file location!
boston = pd.read_csv('../experimental/BostonHousing.csv')

# Let's see what we're working with
print('📋 Dataset Overview:')
print(f'   Rows: {boston.shape[0]} (each row is a neighborhood)')
print(f'   Columns: {boston.shape[1]} (each column is a variable)')
print('\n📊 First 5 rows of data:')
boston.head()

### 📖 Understanding Each Variable

| Variable | Full Name | Description | Type |
|----------|-----------|-------------|------|
| **CRIM** | Crime Rate | Per capita crime rate | Continuous |
| **ZN** | Zoning | % of land zoned for large lots | Continuous |
| **INDUS** | Industry | % of non-retail business acres | Continuous |
| **CHAS** | Charles River | 1 if near river, 0 otherwise | Binary |
| **NOX** | Nitrogen Oxide | Air pollution concentration | Continuous |
| **RM** | Rooms | Average number of rooms | Continuous |
| **AGE** | Age | % of homes built before 1940 | Continuous |
| **DIS** | Distance | Distance to employment centers | Continuous |
| **RAD** | Radial Highways | Accessibility index | Continuous |
| **TAX** | Tax Rate | Property tax per $10,000 | Continuous |
| **PTRATIO** | Pupil-Teacher | Pupil-teacher ratio | Continuous |
| **LSTAT** | Lower Status | % lower socioeconomic status | Continuous |
| **MEDV** | Median Value | **TARGET** - Median home value ($1000s) | Continuous |

In [ ]:
# ============================================
# EXPLORE THE DATA
# ============================================

# Get summary statistics
print('📈 Summary Statistics:')
print('This shows min, max, mean, etc. for each variable\n')
boston.describe().round(2)

In [ ]:
# ============================================
# PREPARE THE DATA
# ============================================

# Define our target variable (what we want to predict)
target = 'MEDV'

# X = all columns EXCEPT the target (these are our predictors)
# We also remove 'CAT. MEDV' if it exists (it's a categorical version)
cols_to_drop = [target]
if 'CAT. MEDV' in boston.columns:
    cols_to_drop.append('CAT. MEDV')

X = boston.drop(cols_to_drop, axis=1)

# y = just the target column
y = boston[target]

print('✅ Data prepared!')
print(f'   X (predictors): {X.shape[0]} samples, {X.shape[1]} features')
print(f'   y (target): {y.shape[0]} values')
print(f'\n📋 Predictor variables: {list(X.columns)}')

---
## 📈 Part 4: Simple Linear Regression (One Predictor)

Let's start simple: predict house price using ONLY the number of rooms.

### The Formula
```
MEDV = β₀ + β₁ × RM
```

In plain English: *"House price = Base price + (Some amount × Number of rooms)"*

In [ ]:
# ============================================
# SIMPLE LINEAR REGRESSION
# ============================================

# Step 1: Select just one predictor (RM = rooms)
X_simple = boston[['RM']]

# Step 2: Add a constant (intercept term)
# This is β₀ in our formula - the base value when RM = 0
X_simple_const = sm.add_constant(X_simple)

# Step 3: Fit the model using OLS (Ordinary Least Squares)
# OLS finds the line that minimizes the sum of squared errors
simple_model = sm.OLS(y, X_simple_const).fit()

# Step 4: View the results
print('=' * 60)
print('SIMPLE LINEAR REGRESSION RESULTS')
print('=' * 60)
print(simple_model.summary())

### 🔍 How to Read the Regression Output

The output above has A LOT of information. Let's break it down:

---

#### **Section 1: Model Information (Top Left)**

| Metric | What It Means |
|--------|---------------|
| **Dep. Variable** | The variable we're predicting (MEDV) |
| **Model** | Type of regression (OLS = Ordinary Least Squares) |
| **No. Observations** | Number of data points used |
| **Df Residuals** | Degrees of freedom for residuals (n - p - 1) |
| **Df Model** | Number of predictors (not counting intercept) |

---

#### **Section 2: Goodness of Fit (Top Right)**

| Metric | What It Means | Good Values |
|--------|---------------|-------------|
| **R-squared** | % of variance in Y explained by the model | 0-1 (higher = better) |
| **Adj. R-squared** | R² adjusted for number of predictors | Use this when comparing models |
| **F-statistic** | Tests if the model is better than just using the mean | Higher = better |
| **Prob (F-statistic)** | p-value for F-test | < 0.05 means model is significant |
| **AIC/BIC** | Information criteria for model comparison | Lower = better |

---

#### **Section 3: Coefficients (Middle Table) - THE MOST IMPORTANT PART!**

| Column | What It Means |
|--------|---------------|
| **coef** | The coefficient value (β) - how much Y changes when X increases by 1 |
| **std err** | Standard error - uncertainty in the coefficient estimate |
| **t** | t-statistic = coef / std err (tests if coef ≠ 0) |
| **P>\|t\|** | p-value - probability coef is actually 0 |
| **[0.025, 0.975]** | 95% confidence interval for the coefficient |

**How to interpret p-values:**
- **p < 0.05**: The variable is statistically significant ✅
- **p ≥ 0.05**: The variable may not be useful ❌

---

#### **Section 4: Diagnostic Tests (Bottom)**

| Test | What It Checks | Good Values |
|------|----------------|-------------|
| **Omnibus** | Are residuals normally distributed? | p > 0.05 |
| **Durbin-Watson** | Is there autocorrelation? | Close to 2 |
| **Jarque-Bera** | Another normality test | p > 0.05 |
| **Skew** | Are residuals symmetric? | Close to 0 |
| **Kurtosis** | Are residuals too peaked/flat? | Close to 3 |

In [ ]:
# ============================================
# INTERPRETING THE COEFFICIENTS
# ============================================

# Extract the coefficients
intercept = simple_model.params['const']
slope = simple_model.params['RM']
r_squared = simple_model.rsquared

print('📊 COEFFICIENT INTERPRETATION')
print('=' * 50)
print(f'\n🔹 Intercept (β₀) = {intercept:.2f}')
print(f'   Meaning: When RM = 0, predicted MEDV = ${intercept:.2f}k')
print(f'   (This is theoretical - no house has 0 rooms!)')

print(f'\n🔹 Slope (β₁) = {slope:.2f}')
print(f'   Meaning: For each additional room, home value')
print(f'   increases by ${slope:.2f}k (${slope*1000:.0f})')

print(f'\n🔹 R-squared = {r_squared:.4f} ({r_squared*100:.1f}%)')
print(f'   Meaning: Number of rooms explains {r_squared*100:.1f}%')
print(f'   of the variation in home prices.')
print(f'   The other {(1-r_squared)*100:.1f}% is due to other factors.')

In [ ]:
# ============================================
# VISUALIZE SIMPLE REGRESSION
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scatter plot with regression line
ax1 = axes[0]
ax1.scatter(boston['RM'], y, alpha=0.5, color='steelblue', label='Actual Data')
ax1.plot(boston['RM'], simple_model.predict(X_simple_const), 
         color='red', linewidth=2, label='Regression Line')
ax1.set_xlabel('Average Number of Rooms (RM)', fontsize=12)
ax1.set_ylabel('Median Home Value ($1000s)', fontsize=12)
ax1.set_title('Simple Linear Regression: Rooms vs Price', fontsize=14)
ax1.legend()

# Plot 2: Residuals
ax2 = axes[1]
residuals = simple_model.resid
ax2.scatter(simple_model.fittedvalues, residuals, alpha=0.5, color='steelblue')
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Fitted Values', fontsize=12)
ax2.set_ylabel('Residuals', fontsize=12)
ax2.set_title('Residual Plot (Should be random around 0)', fontsize=14)

plt.tight_layout()
plt.show()

print('💡 The regression line shows the "best fit" through the data.')
print('💡 Residuals should be randomly scattered - patterns indicate problems!')

---
## 📈 Part 5: Multiple Linear Regression (Many Predictors)

Now let's use ALL available predictors to build a more complete model.

### Why Use Multiple Predictors?
- One variable rarely tells the whole story
- Multiple predictors can explain more variance
- We can control for confounding variables

### The Formula
```
MEDV = β₀ + β₁×CRIM + β₂×ZN + β₃×INDUS + ... + βₙ×LSTAT
```

In [ ]:
# ============================================
# MULTIPLE LINEAR REGRESSION
# ============================================

# Add constant (intercept) to all predictors
X_const = sm.add_constant(X)

# Fit the full model with ALL predictors
full_model = sm.OLS(y, X_const).fit()

print('=' * 60)
print('MULTIPLE LINEAR REGRESSION RESULTS (ALL PREDICTORS)')
print('=' * 60)
print(full_model.summary())

In [ ]:
# ============================================
# INTERPRETING MULTIPLE REGRESSION
# ============================================

print('📊 COEFFICIENT INTERPRETATION (Multiple Regression)')
print('=' * 60)

# Create a nice summary table
coef_df = pd.DataFrame({
    'Coefficient': full_model.params.round(4),
    'Std Error': full_model.bse.round(4),
    'p-value': full_model.pvalues.round(4),
    'Significant?': ['✅ Yes' if p < 0.05 else '❌ No' for p in full_model.pvalues]
})

print(coef_df)

print('\n💡 KEY INSIGHT:')
print(f'   R-squared improved from {simple_model.rsquared:.3f} to {full_model.rsquared:.3f}')
print(f'   We now explain {full_model.rsquared*100:.1f}% of the variance!')
print('\n⚠️  Variables with p-value > 0.05 may not be useful.')

### 🤔 Understanding Coefficient Interpretation in Multiple Regression

**Important:** In multiple regression, each coefficient represents the effect of that variable **while holding all other variables constant**.

For example, if the coefficient for RM is 3.8:
- "For each additional room, home value increases by $3,800..."
- "**...holding crime rate, distance to work, etc. constant**"

This is different from simple regression where we don't control for other factors!

---
## 📏 Part 6: Understanding Regression Metrics

Let's dive deeper into what each metric means and how to use it.

In [ ]:
# ============================================
# CALCULATING AND EXPLAINING METRICS
# ============================================

# Get predictions from our full model
y_pred = full_model.predict(X_const)

# Calculate various metrics
n = len(y)  # Number of observations
p = X.shape[1]  # Number of predictors

# R-squared: Proportion of variance explained
ss_res = np.sum((y - y_pred) ** 2)  # Sum of squared residuals
ss_tot = np.sum((y - y.mean()) ** 2)  # Total sum of squares
r2 = 1 - (ss_res / ss_tot)

# Adjusted R-squared: Penalizes for adding more predictors
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

# Mean Squared Error (MSE)
mse = ss_res / n

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = np.mean(np.abs(y - y_pred))

print('📊 REGRESSION METRICS EXPLAINED')
print('=' * 60)

In [ ]:
# ============================================
# R-SQUARED EXPLANATION
# ============================================

print('\n🔹 R-SQUARED (R²) = {:.4f}'.format(r2))
print('-' * 50)
print('What it measures:')
print('   The proportion of variance in Y explained by the model.')
print('\nFormula:')
print('   R² = 1 - (SS_residual / SS_total)')
print('   R² = 1 - (Unexplained variance / Total variance)')
print('\nInterpretation:')
print(f'   Our model explains {r2*100:.1f}% of the variation in home prices.')
print(f'   The remaining {(1-r2)*100:.1f}% is due to factors not in our model.')
print('\nGuidelines:')
print('   R² = 0.0 - 0.3: Weak model')
print('   R² = 0.3 - 0.6: Moderate model')
print('   R² = 0.6 - 0.9: Strong model')
print('   R² > 0.9: Very strong (but check for overfitting!)')

In [ ]:
# ============================================
# ADJUSTED R-SQUARED EXPLANATION
# ============================================

print('\n🔹 ADJUSTED R-SQUARED = {:.4f}'.format(adj_r2))
print('-' * 50)
print('What it measures:')
print('   R² adjusted for the number of predictors.')
print('\nWhy we need it:')
print('   Regular R² ALWAYS increases when you add more predictors,')
print('   even if they are useless! Adjusted R² penalizes for this.')
print('\nFormula:')
print('   Adj R² = 1 - (1 - R²) × (n - 1) / (n - p - 1)')
print('   where n = observations, p = predictors')
print('\nWhen to use:')
print('   ✅ Use Adjusted R² when comparing models with different')
print('      numbers of predictors.')
print('   ✅ If Adj R² is much lower than R², you may have too many')
print('      useless predictors.')

In [ ]:
# ============================================
# ERROR METRICS EXPLANATION
# ============================================

print('\n🔹 MEAN SQUARED ERROR (MSE) = {:.4f}'.format(mse))
print('-' * 50)
print('What it measures:')
print('   Average of squared differences between predicted and actual.')
print('\nFormula:')
print('   MSE = (1/n) × Σ(y_actual - y_predicted)²')
print('\nPros/Cons:')
print('   ✅ Penalizes large errors more than small ones')
print('   ❌ Units are squared (hard to interpret)')

print('\n🔹 ROOT MEAN SQUARED ERROR (RMSE) = {:.4f}'.format(rmse))
print('-' * 50)
print('What it measures:')
print('   Square root of MSE - in the same units as Y!')
print('\nInterpretation:')
print(f'   On average, our predictions are off by ${rmse:.2f}k (${rmse*1000:.0f})')
print('\nPros/Cons:')
print('   ✅ Same units as the target variable')
print('   ✅ Most commonly used error metric')

print('\n🔹 MEAN ABSOLUTE ERROR (MAE) = {:.4f}'.format(mae))
print('-' * 50)
print('What it measures:')
print('   Average of absolute differences (no squaring).')
print('\nInterpretation:')
print(f'   On average, our predictions are off by ${mae:.2f}k (${mae*1000:.0f})')
print('\nPros/Cons:')
print('   ✅ Easy to interpret')
print('   ✅ Less sensitive to outliers than RMSE')
print('   ❌ Doesn\'t penalize large errors as much')

In [ ]:
# ============================================
# METRICS SUMMARY TABLE
# ============================================

print('\n📊 METRICS SUMMARY')
print('=' * 60)

metrics_df = pd.DataFrame({
    'Metric': ['R-squared', 'Adjusted R²', 'MSE', 'RMSE', 'MAE'],
    'Value': [f'{r2:.4f}', f'{adj_r2:.4f}', f'{mse:.4f}', f'{rmse:.4f}', f'{mae:.4f}'],
    'Interpretation': [
        f'{r2*100:.1f}% variance explained',
        f'{adj_r2*100:.1f}% (adjusted for # predictors)',
        f'Avg squared error',
        f'Avg error: ${rmse*1000:.0f}',
        f'Avg absolute error: ${mae*1000:.0f}'
    ]
})

print(metrics_df.to_string(index=False))

---
## 📈 Part 7: Forward Stepwise Regression

### The Problem with Using All Variables
- Some variables might not be useful
- Too many variables can cause **overfitting**
- We want the **simplest model** that still works well

### Forward Selection: The Idea
Start with nothing, then add variables one at a time:

1. **Start** with no predictors (just the intercept)
2. **Test** each variable - which one improves the model most?
3. **Add** the best one (if it's statistically significant)
4. **Repeat** until no more significant variables can be added

### Why Use Forward Selection?
- Automatically finds the best subset of features
- Avoids including useless variables
- Results in a simpler, more interpretable model

In [ ]:
# ============================================
# FORWARD STEPWISE REGRESSION FUNCTION
# ============================================

def forward_regression(X, y, significance_level=0.05):
    """
    Perform Forward Stepwise Regression.
    
    Parameters:
    -----------
    X : DataFrame
        Predictor variables (features)
    y : Series
        Target variable
    significance_level : float
        p-value threshold for adding variables (default 0.05)
    
    Returns:
    --------
    list : Names of selected features
    
    How it works:
    1. Start with empty model
    2. Try adding each remaining variable
    3. Keep the one with lowest p-value (if < threshold)
    4. Repeat until no more significant variables
    """
    
    # All available features
    initial_features = X.columns.tolist()
    
    # Features we've selected (starts empty)
    best_features = []
    
    print('🔄 FORWARD SELECTION PROCESS')
    print('=' * 50)
    print(f'Starting with {len(initial_features)} candidate features\n')
    
    step = 1
    
    # Keep going until we can't add any more
    while len(initial_features) > 0:
        
        # Features not yet selected
        remaining = list(set(initial_features) - set(best_features))
        
        if len(remaining) == 0:
            break
        
        # Store p-values for each candidate
        new_pval = pd.Series(index=remaining, dtype=float)
        
        # Try adding each remaining feature
        for feature in remaining:
            # Build model with current best + this new feature
            features_to_test = best_features + [feature]
            X_test = sm.add_constant(X[features_to_test])
            model = sm.OLS(y, X_test).fit()
            
            # Get p-value for the new feature
            new_pval[feature] = model.pvalues[feature]
        
        # Find the feature with lowest p-value
        min_p = new_pval.min()
        best_candidate = new_pval.idxmin()
        
        # If significant, add it!
        if min_p < significance_level:
            best_features.append(best_candidate)
            print(f'Step {step}: Added "{best_candidate}"')
            print(f'         p-value = {min_p:.6f} ✅')
            step += 1
        else:
            print(f'\nStopping: Best candidate "{best_candidate}" has p={min_p:.4f} > {significance_level}')
            break
    
    print('\n' + '=' * 50)
    print(f'✅ Selected {len(best_features)} features')
    
    return best_features

In [ ]:
# ============================================
# RUN FORWARD REGRESSION
# ============================================

# Run the forward selection
selected_features = forward_regression(X, y, significance_level=0.05)

print(f'\n📋 Selected features: {selected_features}')

In [ ]:
# ============================================
# FIT FINAL FORWARD MODEL
# ============================================

# Build the final model with selected features
X_forward = sm.add_constant(X[selected_features])
forward_model = sm.OLS(y, X_forward).fit()

print('=' * 60)
print('FORWARD SELECTION MODEL RESULTS')
print('=' * 60)
print(forward_model.summary())

---
## 📈 Part 8: Backward Stepwise Regression

### Backward Elimination: The Opposite Approach

Instead of starting empty and adding, we:

1. **Start** with ALL predictors
2. **Find** the least significant variable (highest p-value)
3. **Remove** it (if p-value > threshold)
4. **Repeat** until all remaining variables are significant

### When to Use Backward vs Forward?

| Method | Best When... |
|--------|-------------|
| **Forward** | You have many variables and expect few to be useful |
| **Backward** | You have fewer variables and want to keep most of them |
| **Both** | Run both and compare results! |

In [ ]:
# ============================================
# BACKWARD STEPWISE REGRESSION FUNCTION
# ============================================

def backward_regression(X, y, significance_level=0.05):
    """
    Perform Backward Stepwise Regression.
    
    Parameters:
    -----------
    X : DataFrame
        Predictor variables (features)
    y : Series
        Target variable
    significance_level : float
        p-value threshold for keeping variables (default 0.05)
    
    Returns:
    --------
    list : Names of remaining features
    
    How it works:
    1. Start with ALL variables
    2. Find the one with highest p-value
    3. Remove it if p-value > threshold
    4. Repeat until all remaining are significant
    """
    
    # Start with ALL features
    features = X.columns.tolist()
    
    print('🔄 BACKWARD ELIMINATION PROCESS')
    print('=' * 50)
    print(f'Starting with {len(features)} features\n')
    
    step = 1
    
    while len(features) > 0:
        # Fit model with current features
        X_with_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_with_const).fit()
        
        # Get p-values (exclude the constant/intercept)
        p_values = model.pvalues.drop('const')
        
        # Find the highest p-value
        max_p = p_values.max()
        worst_feature = p_values.idxmax()
        
        # If not significant, remove it
        if max_p > significance_level:
            features.remove(worst_feature)
            print(f'Step {step}: Removed "{worst_feature}"')
            print(f'         p-value = {max_p:.6f} ❌')
            step += 1
        else:
            print(f'\nStopping: All remaining variables are significant (p < {significance_level})')
            break
    
    print('\n' + '=' * 50)
    print(f'✅ Kept {len(features)} features')
    
    return features

In [ ]:
# ============================================
# RUN BACKWARD REGRESSION
# ============================================

# Run the backward elimination
backward_features = backward_regression(X, y, significance_level=0.05)

print(f'\n📋 Remaining features: {backward_features}')

In [ ]:
# ============================================
# FIT FINAL BACKWARD MODEL
# ============================================

# Build the final model with remaining features
X_backward = sm.add_constant(X[backward_features])
backward_model = sm.OLS(y, X_backward).fit()

print('=' * 60)
print('BACKWARD ELIMINATION MODEL RESULTS')
print('=' * 60)
print(backward_model.summary())

---
## 📊 Part 9: Comparing All Models

Let's compare all the models we've built to see which performs best!

In [ ]:
# ============================================
# MODEL COMPARISON
# ============================================

print('📊 MODEL COMPARISON')
print('=' * 70)

# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Simple (RM only)', 'Full (all features)', 'Forward Selection', 'Backward Elimination'],
    'Num Features': [1, X.shape[1], len(selected_features), len(backward_features)],
    'R-squared': [
        simple_model.rsquared,
        full_model.rsquared,
        forward_model.rsquared,
        backward_model.rsquared
    ],
    'Adj R-squared': [
        simple_model.rsquared_adj,
        full_model.rsquared_adj,
        forward_model.rsquared_adj,
        backward_model.rsquared_adj
    ],
    'AIC': [
        simple_model.aic,
        full_model.aic,
        forward_model.aic,
        backward_model.aic
    ]
})

comparison['R-squared'] = comparison['R-squared'].round(4)
comparison['Adj R-squared'] = comparison['Adj R-squared'].round(4)
comparison['AIC'] = comparison['AIC'].round(1)

print(comparison.to_string(index=False))

print('\n💡 INTERPRETATION:')
print('   - Higher R² and Adj R² = better fit')
print('   - Lower AIC = better model (penalizes complexity)')
print('   - Fewer features = simpler, more interpretable model')

---
## 🔍 Part 10: Checking for Multicollinearity (VIF)

### What is Multicollinearity?

**Multicollinearity** occurs when predictor variables are highly correlated with each other.

**Why is it a problem?**
- Makes coefficients unstable and hard to interpret
- Inflates standard errors
- Can make significant variables appear insignificant

### VIF (Variance Inflation Factor)

VIF measures how much the variance of a coefficient is "inflated" due to correlation with other predictors.

| VIF Value | Interpretation |
|-----------|----------------|
| 1 | No correlation with other predictors |
| 1-5 | Low correlation (acceptable) ✅ |
| 5-10 | Moderate correlation (investigate) ⚠️ |
| > 10 | High correlation (problem!) ❌ |

In [ ]:
# ============================================
# CALCULATE VIF
# ============================================

# Calculate VIF for each predictor
X_with_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data['Feature'] = X_with_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_with_const.values, i) 
                   for i in range(X_with_const.shape[1])]

# Add interpretation
def interpret_vif(vif):
    if vif == np.inf:
        return '⚠️ Infinite (perfect correlation)'
    elif vif > 10:
        return '❌ High (problem!)'
    elif vif > 5:
        return '⚠️ Moderate (investigate)'
    else:
        return '✅ Low (OK)'

vif_data['Interpretation'] = vif_data['VIF'].apply(interpret_vif)
vif_data['VIF'] = vif_data['VIF'].round(2)

print('📊 VARIANCE INFLATION FACTORS')
print('=' * 60)
print(vif_data.to_string(index=False))

print('\n💡 Variables with high VIF may need to be removed or combined.')

---
## 📉 Part 11: Residual Analysis

### What are Residuals?

**Residual** = Actual value - Predicted value

Residuals tell us how far off our predictions are.

### What Should Good Residuals Look Like?

1. **Randomly scattered** around zero (no patterns)
2. **Normally distributed** (bell curve shape)
3. **Constant variance** (same spread at all prediction levels)

If residuals show patterns, our model is missing something!

In [ ]:
# ============================================
# RESIDUAL ANALYSIS PLOTS
# ============================================

# Use the forward model for analysis
residuals = forward_model.resid
fitted_values = forward_model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Fitted Values
ax1 = axes[0, 0]
ax1.scatter(fitted_values, residuals, alpha=0.5, color='steelblue')
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax1.set_xlabel('Fitted Values')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted Values\n(Should be random around 0)')

# Plot 2: Histogram of Residuals
ax2 = axes[0, 1]
ax2.hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residuals')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Residuals\n(Should be bell-shaped)')

# Plot 3: Q-Q Plot
ax3 = axes[1, 0]
from scipy import stats
stats.probplot(residuals, dist='norm', plot=ax3)
ax3.set_title('Q-Q Plot\n(Points should follow the line)')

# Plot 4: Actual vs Predicted
ax4 = axes[1, 1]
ax4.scatter(y, fitted_values, alpha=0.5, color='steelblue')
ax4.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
ax4.set_xlabel('Actual Values')
ax4.set_ylabel('Predicted Values')
ax4.set_title('Actual vs Predicted\n(Points should follow the line)')

plt.tight_layout()
plt.show()

---
## ✅ Part 12: Cross-Validation

### Why Cross-Validation?

A model might perform well on the data it was trained on, but poorly on new data. This is called **overfitting**.

**Cross-validation** tests how well our model generalizes to unseen data.

### K-Fold Cross-Validation

1. Split data into K equal parts ("folds")
2. Train on K-1 folds, test on the remaining fold
3. Repeat K times (each fold gets to be the test set once)
4. Average the results

```
Fold 1: [TEST] [Train] [Train] [Train] [Train]
Fold 2: [Train] [TEST] [Train] [Train] [Train]
Fold 3: [Train] [Train] [TEST] [Train] [Train]
Fold 4: [Train] [Train] [Train] [TEST] [Train]
Fold 5: [Train] [Train] [Train] [Train] [TEST]
```

In [ ]:
# ============================================
# CROSS-VALIDATION
# ============================================

# Use sklearn for cross-validation
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Create sklearn model with selected features
sklearn_model = LinearRegression()

# Perform 10-fold cross-validation
cv_scores = cross_val_score(
    sklearn_model, 
    X[selected_features], 
    y, 
    cv=10,  # 10 folds
    scoring='r2'  # Use R² as the metric
)

print('📊 10-FOLD CROSS-VALIDATION RESULTS')
print('=' * 50)
print(f'\nR² scores for each fold:')
for i, score in enumerate(cv_scores, 1):
    print(f'   Fold {i:2d}: {score:.4f}')

print(f'\n📈 Summary:')
print(f'   Mean R²: {cv_scores.mean():.4f}')
print(f'   Std Dev: {cv_scores.std():.4f}')
print(f'   Min R²:  {cv_scores.min():.4f}')
print(f'   Max R²:  {cv_scores.max():.4f}')

print('\n💡 INTERPRETATION:')
print('   - Mean R² tells us expected performance on new data')
print('   - Low std dev means consistent performance across folds')
print('   - If CV R² << training R², model may be overfitting')

---
## 📝 Part 13: Summary and Key Takeaways

### What We Learned

#### 1. Types of Regression

| Type | Description | When to Use |
|------|-------------|-------------|
| **Simple** | One predictor | Quick analysis, understanding single relationships |
| **Multiple** | All predictors | When you want to use all available information |
| **Forward** | Add features one by one | Many features, expect few to be useful |
| **Backward** | Remove features one by one | Fewer features, want to keep most |

#### 2. Key Metrics

| Metric | What It Tells You | Good Values |
|--------|-------------------|-------------|
| **R²** | % variance explained | Higher is better (0-1) |
| **Adj R²** | R² adjusted for # predictors | Use for model comparison |
| **p-value** | Is coefficient significant? | < 0.05 |
| **RMSE** | Average prediction error | Lower is better |
| **VIF** | Multicollinearity check | < 5 is good |

#### 3. Best Practices

1. ✅ Always check residual plots
2. ✅ Use cross-validation to test generalization
3. ✅ Check for multicollinearity (VIF)
4. ✅ Prefer simpler models when performance is similar
5. ✅ Look at Adjusted R² when comparing models
6. ❌ Don't just maximize R² (can lead to overfitting)
7. ❌ Don't include variables with p > 0.05

In [ ]:
# ============================================
# FINAL SUMMARY
# ============================================

print('🎓 LESSON COMPLETE!')
print('=' * 60)
print('\nYou now know how to:')
print('   ✅ Build simple and multiple regression models')
print('   ✅ Interpret coefficients and p-values')
print('   ✅ Understand R², Adjusted R², RMSE, and other metrics')
print('   ✅ Use forward and backward selection')
print('   ✅ Check for multicollinearity with VIF')
print('   ✅ Analyze residuals for model diagnostics')
print('   ✅ Validate models with cross-validation')
print('\n🚀 Next steps:')
print('   - Try this with your own dataset!')
print('   - Explore regularized regression (Ridge, Lasso)')
print('   - Learn about polynomial regression for non-linear relationships')